In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from selenium import webdriver

import pandas as pd

from time import sleep

import datetime

from bs4 import BeautifulSoup

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

import os

from selenium.webdriver.chrome.service import Service as ChromeService
import requests
import urllib3
import re
from urllib.parse import urljoin

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException


In [2]:


# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'PT BPOR' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.3")

now=datetime.datetime.now()

filename = '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)


tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_argument("--disable-search-engine-choice-screen")

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



Running PT BPOR Web Scraping Tool v.1.3


In [3]:


# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        'PT BPOR 29': 'https://www.bportugal.pt/en/entidades-autorizadas',

		}



Typology={

        'PT BPOR 29': 'Authorised institutions',

		}



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}



label_row = ['Type of institution', 'Institution Acronym', 'Abbreviation', 'Status', 'Address', 'Foreign Address', 'Location', 'Foreign Location', 'Postal Code', 'Foreign Postal Code', 'Country', 'Foreign Country', 'Start date of activity', 'Name', 'IF Code' ]

# label_row_PT = ['Tipo', 'Sigla da Instituição', 'Estado', 'Morada', 'Morada Estrangeiro', 'Localidade', 'Localidade Estrangeiro', 'Código Postal', 'Código Postal Estrangeiro', 'País', 'País estrangeiro', 'Data de Início de Atividade', 'Denominação da Sede', 'Código de IF' ]

label_sqldict = ['Typology', 'InternalID_2', 'InternalID_2', 'RegulationType', 'Address_1', 'Address_1', 'City', 'City', 'Zip', 'Zip', 'Cntry', 'Cntry', 'RegulationDate', 'Name', 'InternalID_1' ]



processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def click_on_cookies(Msg_accept):

    try:

        driver.find_element(By.XPATH,f'//*[@id="onetrust-accept-btn-handler"]').click()

        print(f'[INFO] : click "{Msg_accept}" button')

        sleep(1)

    except Exception as err:

        print(f'[ERROR] : Failed to click "{Msg_accept}" button on the cookies banner:\n {err}')





In [ ]:
# %%
#------------------------------------------------ Begin_Main ----------------------------------------

import requests
import urllib3
import re
from urllib.parse import urljoin

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://www.bportugal.pt"
LIST_URL = "https://www.bportugal.pt/en/page/list-authorised-institutions"

def clean_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

def get_h1_company_name(soup):
    h1 = soup.find("h1")
    return clean_text(h1.get_text(" ", strip=True)) if h1 else ""

def get_detail_values(soup):
    values = {}

    rows = soup.select("div.row.bdpsi--paragraph--row div.col-12.col-lg-4")
    for row in rows:
        divs = row.find_all("div", recursive=False)

        if len(divs) >= 2:
            label = clean_text(divs[0].get_text(" ", strip=True))
            value = clean_text(divs[1].get_text(" ", strip=True))
            if label:
                values[label] = value
            continue

        all_divs = row.find_all("div")
        texts = [clean_text(div.get_text(" ", strip=True)) for div in all_divs]
        texts = [text for text in texts if text]

        if len(texts) >= 2:
            values[texts[0]] = texts[1]

    return values

def build_session_from_driver(driver):
    session = requests.Session()
    session.headers.update({
        "User-Agent": driver.execute_script("return navigator.userAgent;"),
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": BASE_URL + "/",
    })

    for cookie in driver.get_cookies():
        session.cookies.set(cookie["name"], cookie["value"])

    return session

def extract_links_from_soup(soup):
    links = {}
    seen = set()

    for a in soup.select('div.content-container--text-container a.content-container--link-container[href]'):
        href = clean_text(a.get("href", ""))
        if not href:
            continue

        full_link = urljoin(BASE_URL, href)
        company_name = clean_text(a.get_text(" ", strip=True))
        if full_link not in seen and company_name:
            seen.add(full_link)
            links.setdefault(company_name, [])
            links[company_name].append(full_link)

    return links

for k, reg in enumerate(regdict):

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_")
    driver.get(regdict[reg])
    sleep(5)

    try:
        cookie_bannier = driver.find_element(By.XPATH, '//*[@id="onetrust-banner-sdk"]')
        cookie_is_displayed = cookie_bannier.is_displayed()
    except:
        cookie_is_displayed = False

    if cookie_is_displayed:
        click_on_cookies("Accept All Cookies")
        sleep(2)

    items = driver.find_element(By.CSS_SELECTOR, "div.list-items-indexed--count.bdpsi-label-small-highcase").text

    items = int(items.split()[0])

    total_company = 0
    preview_total_company = 0
    scrollTo = 0
    stagnant_rounds = 0
    repeated_batch_rounds = 0
    max_stagnant_rounds = 10
    max_repeated_batch_rounds = 30
    max_scroll_rounds = max(10, items)
    last_batch_signature = None
    links = {}
    seen_links = set()

    while total_company < items:
        if scrollTo >= max_scroll_rounds:
            print(f"[WARNING] : reached max scroll rounds ({max_scroll_rounds}) with {total_company}/{items} companies")
            break

        scrollTo += 1
        cards = driver.find_elements(By.CSS_SELECTOR, "div.content-container--text-container a.content-container--link-container")
        current_links = {}
        new_links_this_round = 0

        for card in cards:
            href = card.get_attribute("href")
            company_name = clean_text(card.text)
            if href and company_name:
                current_links.setdefault(company_name, [])
                if href not in current_links[company_name]:
                    current_links[company_name].append(href)

        current_batch_signature = tuple(
            sorted((name, tuple(sorted(company_links))) for name, company_links in current_links.items())
        )
        if current_batch_signature == last_batch_signature:
            repeated_batch_rounds += 1
        else:
            repeated_batch_rounds = 0
        last_batch_signature = current_batch_signature

        for company_name, company_links in current_links.items():
            links.setdefault(company_name, [])
            for link in company_links:
                if link not in seen_links:
                    seen_links.add(link)
                    links[company_name].append(link)
                    new_links_this_round += 1

        total_company = len(seen_links)

        print(f"[INFO] : Total_company = {total_company}/{items}")

        if new_links_this_round == 0:
            stagnant_rounds += 1
        else:
            stagnant_rounds = 0

        if total_company >= items:
            break

        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
        sleep(2)

        if stagnant_rounds >= max_stagnant_rounds:
            print(f"[WARNING] : no new unique companies after {stagnant_rounds} scrolls")
            break

        if repeated_batch_rounds >= max_repeated_batch_rounds:
            print(f"[WARNING] : same batch repeated {repeated_batch_rounds} times, stopping scroll")
            break


        preview_total_company = total_company



company_link_pairs = [
    (company_name, link)
    for company_name, company_links in links.items()
    for link in company_links
]

for company_name, link in company_link_pairs[3:]:
    page_loaded = False
    values = {}
    for attempt in range(3):
        try:
            driver.get(link)

            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "h1"))
            )

            title = driver.find_element(By.TAG_NAME, "h1")

            if title:
                # print(f"[INFO] : Working on {company_name}")
                page_loaded = True
                break
        except (TimeoutException, WebDriverException, Exception) as err:
            print(f"[WARNING] : attempt {attempt+1}/3 failed for {link} | {err}")

            if attempt == 0:
                try:
                    driver.refresh()
                    sleep(3)
                except:
                    pass

            elif attempt == 1:
                try:
                    driver.quit()
                except:
                    pass

                sleep(5)
                driver = webdriver.Chrome(options=chromeOptions)
                driver.maximize_window()

            else:
                print(f"[ERROR] : could not load page -> {link}")

    if not page_loaded:
        continue

    try:
        infor_content = driver.find_element(By.ID, "block-bportugal-content")
        rows = infor_content.find_elements(By.XPATH, ".//div[@class='row bdpsi--paragraph--row']")
        for row in rows:
            # print(row)
            
            divs = row.find_elements(By.XPATH, "./div")
            if len(divs) < 2:
                continue

            # label = clean_text(divs[0].text)
            # value = clean_text(divs[1].text)
            for i, div in enumerate(divs):
                label = div.text.split("\n")[0]
                value = div.text.split("\n")[1] if len(div.text.split("\n")) > 1 else ""
                print(label, ":", value)
                values[label] = value
        sqldict["Name"].append(' '.join(company_name.split(' ')[3:]))
        sqldict["Typology"].append(values.get("TYPE OF INSTITUTION", ""))
        sqldict["RegulationType"].append(values.get("STATUS", "").replace("Active", "Regulated"))
        sqldict["InternalID_1"].append(values.get("IF CODE", ""))
        sqldict["InternalID_1_type"].append("IF CODE" if values.get("IF CODE", "") else "")
        sqldict["Address_1"].append(values.get("ADDRESS", ""))
        sqldict["Zip"].append(values.get("POSTAL CODE", ""))
        sqldict["City"].append(values.get("LOCATION", ""))
        sqldict["Cntry"].append(values.get("COUNTRY", ""))
        sqldict["RegulationDate"].append(values.get("START DATE OF ACTIVITY", ""))
        sqldict["ListProcessDate"].append(processdate)
        sqldict['RegCtry'].append(reg.split()[0])
        sqldict['RegCode'].append(reg.split()[1])
        sqldict['ListCode'].append(reg.split()[2])
        sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
    except Exception as err:
        print(f"[WARNING] : failed extracting detail values -> {link} | {err}")

[INFO] : Working 1/1 _(PT BPOR 29)_
[INFO] : click "Accept All Cookies" button
[INFO] : Total_company = 20/1391
[INFO] : Total_company = 40/1391
[INFO] : Total_company = 80/1391
[INFO] : Total_company = 80/1391
[INFO] : Total_company = 80/1391
[INFO] : Total_company = 100/1391
[INFO] : Total_company = 120/1391
[INFO] : Total_company = 160/1391
[INFO] : Total_company = 180/1391
[INFO] : Total_company = 220/1391
[INFO] : Total_company = 260/1391
[INFO] : Total_company = 300/1391
[INFO] : Total_company = 340/1391
[INFO] : Total_company = 380/1391
[INFO] : Total_company = 400/1391
[INFO] : Total_company = 460/1391
[INFO] : Total_company = 480/1391
[INFO] : Total_company = 500/1391
[INFO] : Total_company = 540/1391
[INFO] : Total_company = 560/1391
[INFO] : Total_company = 600/1391
[INFO] : Total_company = 660/1391
[INFO] : Total_company = 660/1391
[INFO] : Total_company = 720/1391
[INFO] : Total_company = 760/1391
[INFO] : Total_company = 800/1391
[INFO] : Total_company = 820/1391
[INFO] :

In [1]:
' '.join(company_name.split(' ')[3:])


NameError: name 'company_name' is not defined

In [5]:



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(filename, index=False)

driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)

sleep(3)
    
    